****
### Import Snowpark and create Snowpark session
****

In [ ]:
import snowflake.snowpark as snowpark
from snowflake.snowpark.functions import month,year,col,sum

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

In [ ]:
-- Using Warehouse, Database, and Schema created during Setup
USE WAREHOUSE snowpark_demo_wh;
USE DATABASE snowpark_demo_db;
USE SCHEMA snowpark_demo_schema;

****
### Load campaign_spend and monthly_revenue tables into Snowpark dataframes
****

In [ ]:
snow_df_spend = session.table('campaign_spend')
display(snow_df_spend)

In [ ]:
snow_df_revenue = session.table('monthly_revenue')
display(snow_df_revenue)

****
### Total Spend per Year and Month For All Channels
****
Let's transform the campaign spend data so we can see total cost per year/month per channel using group_by() and agg() Snowpark DataFrame functions.

In [ ]:
snow_df_spend_per_channel = snow_df_spend.group_by(year('Date'),month('Date'),'channel')\
                                         .agg(sum('total_cost').as_('Total_Cost'))\
                                         .with_column_renamed('"YEAR(DATE)"',"Year")\
                                         .with_column_renamed('"MONTH(DATE)"',"Month")\
                                         .sort('Year','Month')

snow_df_spend_per_channel

****
### Total Spend per Year and Month
****
Let's further transform the campaign spend data by pivoting on the channel dimension. This should give us the campaign spend for every month across all channels on the same row.

In [ ]:
snow_df_spend_per_month = snow_df_spend_per_channel.pivot('channel',['search_engine','social_media','video','email'])\
                                                   .sum('total_cost')\
                                                   .sort('year','month')

snow_df_spend_per_month

snow_df_spend_per_month = snow_df_spend_per_month.select(
                          col("YEAR"),
        col("MONTH"),
        col("'search_engine'").as_("SEARCH_ENGINE"),
        col("'social_media'").as_("SOCIAL_MEDIA"),
        col("'video'").as_("VIDEO"),
        col("'email'").as_("EMAIL")                         
)

snow_df_spend_per_month

****
### Total Revenue per Year and Month
****
Now let's transform the revenue data into revenue per year/month using group_by() and agg() functions.

In [ ]:
snow_df_revenue_per_month = snow_df_revenue.group_by('year','month')\
                                           .agg(sum('revenue'))\
                                           .sort('year','month')\
                                           .with_column_renamed('"SUM(REVENUE)"','REVENUE')

print("Total Revenue per Year and Month")
snow_df_revenue_per_month.show()

****
### Join Total Spend and Total Revenue per Year and Month Across All Channels
****
Next let's join this revenue data with the transformed campaign spend data so we can analyze the spend and revenue data side by side.

In [ ]:
snow_df_spend_and_revenue_per_month = snow_df_spend_per_month.join(snow_df_revenue_per_month,["YEAR","MONTH"])

snow_df_spend_and_revenue_per_month

****
### Examine DataFrame Explain Plan
****
Snowpark makes it really convenient to look at the DataFrame query and execution plan using explain() Snowpark DataFrame function.

In [ ]:
snow_df_spend_and_revenue_per_month.explain()

****
### Save Transformed Data into Snowflake Table
****
Let's save the transformed data into a Snowflake table SPEND_AND_REVENUE_PER_MONTH

In [ ]:
snow_df_spend_and_revenue_per_month.write.mode('overwrite').save_as_table('SPEND_AND_REVENUE_PER_MONTH')

****
### Deploy As Stored Procedure
****
In Snowflake Workspaces (Notebooks), there is no direct "Deploy as Procedure" button like there was in the classic Python Worksheets.

To deploy your notebook code as a stored procedure, you would need to do it manually via SQL using CREATE PROCEDURE

In [ ]:
create or replace procedure campaign_spend_monthly_revenue_data_pipeline_sp()
returns table()
language python
runtime_version = '3.12'
packages = ('snowflake-snowpark-python')
handler = 'main'
as
$$
from snowflake.snowpark.functions import month, year, col, sum

def main(session):
    snow_df_spend = session.table('campaign_spend')

    snow_df_spend_per_channel = snow_df_spend.group_by(year('Date'), month('Date'), 'channel') \
        .agg(sum('total_cost').as_('Total_Cost')) \
        .with_column_renamed('"YEAR(DATE)"', 'Year') \
        .with_column_renamed('"MONTH(DATE)"', 'Month') \
        .sort('Year', 'Month')

    snow_df_spend_per_month = snow_df_spend_per_channel.pivot('channel', ['search_engine', 'social_media', 'video', 'email']) \
        .sum('total_cost') \
        .sort('year', 'month')

    snow_df_spend_per_month = snow_df_spend_per_month.select(
        col('YEAR'),
        col('MONTH'),
        col("'search_engine'").as_('SEARCH_ENGINE'),
        col("'social_media'").as_('SOCIAL_MEDIA'),
        col("'video'").as_('VIDEO'),
        col("'email'").as_('EMAIL')
    )

    snow_df_revenue = session.table('monthly_revenue')

    snow_df_revenue_per_month = snow_df_revenue.group_by('year', 'month') \
        .agg(sum('revenue')) \
        .sort('year', 'month') \
        .with_column_renamed('"SUM(REVENUE)"', 'REVENUE')

    snow_df_spend_and_revenue_per_month = snow_df_spend_per_month.join(snow_df_revenue_per_month, ['YEAR', 'MONTH'])

    snow_df_spend_and_revenue_per_month.write.mode('overwrite').save_as_table('SPEND_AND_REVENUE_PER_MONTH')

    return snow_df_spend_and_revenue_per_month
$$;